### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="marketing_campaign",
    dataset_year="2020",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/rodsaldanha/arketing-campaign",
    download_description="""
We download the data from the Kaggle repository and uzip it to a predefined folder.

mkdir -p local-data-warehouse/marketing_campaign/ && cd local-data-warehouse/marketing_campaign/ && kaggle datasets download rodsaldanha/arketing-campaign && cd ../../ && unzip local-data-warehouse/marketing_campaign/arketing-campaign.zip -d local-data-warehouse/marketing_campaign/ && rm local-data-warehouse/marketing_campaign/arketing-campaign.zip && rm local-data-warehouse/marketing_campaign/marketing_campaign.xlsx
""",
    # References
    academic_reference_bibtex=r"""@misc{saldanha2020marketing,
  author       = {Saldanha, Rodolfo},
  title        = {Marketing Campaign},
  year         = {2020},
  howpublished = {\url{https://www.kaggle.com/datasets/rodsaldanha/arketing-campaign}},
  note         = {Kaggle dataset},
}
""",
    academic_reference_bibtex_key="saldanha2020marketing",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We ensure Dt_Customer is a pandas datetime.
- We drop the ID column.
- We rename the values of the target variable to be more descriptive.
- We drop constant columns.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Response",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Response",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/marketing_campaign.csv", sep=";")
df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"], inplace=True)


cat_features = [
    "AcceptedCmp1",
    "AcceptedCmp2",
    "AcceptedCmp3",
    "AcceptedCmp4",
    "AcceptedCmp5",
    "Marital_Status",
    "Education",
    "Response",
]

df[cat_features] = df[cat_features].astype("category")


df["Response"] = df["Response"].map({0: 'No', 1: 'Yes'})
df["AcceptedCmp1"] = df["AcceptedCmp1"].map({0: 'No', 1: 'Yes'})
df["AcceptedCmp2"] = df["AcceptedCmp2"].map({0: 'No', 1: 'Yes'})
df["AcceptedCmp3"] = df["AcceptedCmp3"].map({0: 'No', 1: 'Yes'})
df["AcceptedCmp4"] = df["AcceptedCmp4"].map({0: 'No', 1: 'Yes'})
df["AcceptedCmp5"] = df["AcceptedCmp5"].map({0: 'No', 1: 'Yes'})

df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%Y-%m-%d")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()